Recimibos un archivo para su analisis, debido a una reunion de emergencia del Rector, nos pidio datos generales del archivo que abordaremos maniana para determinar los alcances especificos en base a los primeros hallazgos

Utilizaremos pyspark para realizar el analisis, por tanto tendremos que crear una sesion con SparkSession.builder

In [6]:
#Importar las librerias
from pyspark.sql import SparkSession

#Creamos la sesion, por convencion la guardamos en una variable llamada spark
spark= SparkSession.builder \
    .appName('analisis_estudiantes') \
    .getOrCreate()

Guardaremos el archivo en una variable, ya que sera sobre un dataframe lo llamaremos df, de momento no sabemos si incluye el encabezado, por tanto, consideraremos todas las filas como datos y revisaremos los primeros datos para inferir si incluyen encabezado

In [15]:
#Guardamos el archivo en un dataframe y mostramos las primeras 5 filas
df= spark.read.csv('StudentPerformance.csv', header='false', inferSchema='true')
df.show(5)

+--------------------+
|                 _c0|
+--------------------+
|index;race_ethnic...|
|0;group B;bachelo...|
|1;group C;some co...|
|2;group B;master'...|
|3;group A;associa...|
+--------------------+
only showing top 5 rows


Al observarlo notamos que si incluye los encabezados, asi que realizamos el ajuste leyendo nuevamente el archivo con header true.
Ademas observamos que el separador no es estandar con ',' se utiliza ';', tambien ajustamos esto con el parametro sep

In [17]:
#Abrimos el archivo correctamente con header true
df= spark.read.csv('StudentPerformance.csv', header='true', inferSchema=True, sep=';')
df.show(5)

+-----+--------------+------------------+------------+-----------------------+---------------+------------------+------------------+---+
|index|race_ethnicity|parental_education|       lunch|test_preparation_course|math_percentage|reading_percentage|writing_percentage|sex|
+-----+--------------+------------------+------------+-----------------------+---------------+------------------+------------------+---+
|    0|       group B| bachelor's degree|    standard|                   none|           0.72|              0.72|              0.74|  F|
|    1|       group C|      some college|    standard|              completed|           0.69|               0.9|              0.88|  F|
|    2|       group B|   master's degree|    standard|                   none|            0.9|              0.95|              0.93|  F|
|    3|       group A|associate's degree|free/reduced|                   none|           0.47|              0.57|              0.44|  M|
|    4|       group C|      some college|

Ya tenemos informacion relevante, podemos inferir el contenido gracias al nombre de las columnas pero como complemento podemos ver los tipos de datos para conocer el tipo de operaciones que pudieramos realizar

In [18]:
#Utilizaremos printschema para ver los tipos de datos, recordando que usando inferSchema True, se ha delegado a pysapark la eleccion de datos
df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- race_ethnicity: string (nullable = true)
 |-- parental_education: string (nullable = true)
 |-- lunch: string (nullable = true)
 |-- test_preparation_course: string (nullable = true)
 |-- math_percentage: double (nullable = true)
 |-- reading_percentage: double (nullable = true)
 |-- writing_percentage: double (nullable = true)
 |-- sex: string (nullable = true)



Ya que hemos trabajado con algunas vistas previas y el esquema de datos, desconocemos la cantidad de registros, es importante tener este dato y podriamos almacenarlo para realizar algunas operaciones

In [19]:
#Guardamos en una variable la cantidad de registros
total_registros= df.count()
print(f'El dataset contiene {total_registros} registros')

El dataset contiene 1000 registros


Pyspark (tambien pandas) cuenta con una funcion interesante para analisis exploratorio, llamada .describe la cual en las columnas que sea posible nos entrega el promedio, el minimo, el maximo y la desviacion estandar

In [20]:
#Utilizamos .describe para obtener un analisis exploratorio simple
df.describe().show()

26/05/17 19:38:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+--------------+------------------+------------+-----------------------+-------------------+-------------------+------------------+----+
|summary|            index|race_ethnicity|parental_education|       lunch|test_preparation_course|    math_percentage| reading_percentage|writing_percentage| sex|
+-------+-----------------+--------------+------------------+------------+-----------------------+-------------------+-------------------+------------------+----+
|  count|             1000|          1000|              1000|        1000|                   1000|               1000|               1000|              1000|1000|
|   mean|            499.5|          NULL|              NULL|        NULL|                   NULL|  0.660890000000001| 0.6916900000000004|0.6805400000000015|NULL|
| stddev|288.8194360957494|          NULL|              NULL|        NULL|                   NULL|0.15163080096009468|0.14600191937252227|0.1519565701086965|NULL|
|    min|             

Gracias a esto podemos saber en las columnas math_percentage, reading_percentage y writing_percentage que se refieren en una escala del 0 al 1, si bien estan en formato double o flotante, para cualquier presentacion de los datos es mejor presentarlo en entero, multiplicandolo por 100 y podriamos iniciar con dos decimales para el detalle, despues agregariamos el signo '%'

En muchas ocasiones recibiremos conjuntos con datos nulos, dependiendo de la situacion variara la decision con respecto a ellos, ya que podemos estandarizarlos (que tomen un valor como 0), ignorarlos para la operacion que corresponda, o incluso eliminarlos y dependiendo de la cantidad de nulos podrian eliminarse las filas o inclusive las columnas completas del conjunto de datos

In [21]:
#Iterar por las columnas y mostrar un mensaje con la cantidad de nulos por cada columna
for columna in df.columns:
    nulos=df.filter(df[columna].isNull()).count()
    print(f'La columna {columna} tiene {nulos} datos nulos')

La columna index tiene 0 datos nulos
La columna race_ethnicity tiene 0 datos nulos
La columna parental_education tiene 0 datos nulos
La columna lunch tiene 0 datos nulos
La columna test_preparation_course tiene 0 datos nulos
La columna math_percentage tiene 0 datos nulos
La columna reading_percentage tiene 0 datos nulos
La columna writing_percentage tiene 0 datos nulos
La columna sex tiene 0 datos nulos


Un dato que pudiera ser interesante podria ser el promedio general, para ello agregaremos una columna llamada general average, sumando los valores de las 3 columnas y dividiendolo entre 3 (numero de columnas)

In [23]:
from pyspark.sql.functions import col
#Calcular el promedio general de los estudiantes y agregarlo en una nueva columna
df = df.withColumn("general_average", (col("math_percentage") + col("reading_percentage") + col("writing_percentage")) / 3)
df.show(5)


+-----+--------------+------------------+------------+-----------------------+---------------+------------------+------------------+---+-------------------+
|index|race_ethnicity|parental_education|       lunch|test_preparation_course|math_percentage|reading_percentage|writing_percentage|sex|    general_average|
+-----+--------------+------------------+------------+-----------------------+---------------+------------------+------------------+---+-------------------+
|    0|       group B| bachelor's degree|    standard|                   none|           0.72|              0.72|              0.74|  F| 0.7266666666666666|
|    1|       group C|      some college|    standard|              completed|           0.69|               0.9|              0.88|  F| 0.8233333333333333|
|    2|       group B|   master's degree|    standard|                   none|            0.9|              0.95|              0.93|  F| 0.9266666666666667|
|    3|       group A|associate's degree|free/reduced|    

Podemos utilizar nuevamente describe para ver los datos de forma general, que tanto habra afectado el promedio general a los maximos y minimos del resto de materias

In [24]:
#Utilizamos describe para ver los resultados
df.describe().show()


+-------+-----------------+--------------+------------------+------------+-----------------------+-------------------+-------------------+------------------+----+-------------------+
|summary|            index|race_ethnicity|parental_education|       lunch|test_preparation_course|    math_percentage| reading_percentage|writing_percentage| sex|    general_average|
+-------+-----------------+--------------+------------------+------------+-----------------------+-------------------+-------------------+------------------+----+-------------------+
|  count|             1000|          1000|              1000|        1000|                   1000|               1000|               1000|              1000|1000|               1000|
|   mean|            499.5|          NULL|              NULL|        NULL|                   NULL|  0.660890000000001| 0.6916900000000004|0.6805400000000015|NULL| 0.6777066666666668|
| stddev|288.8194360957494|          NULL|              NULL|        NULL|           

Observamos que no hay especial varianza entre el promedio general y los promedios de las diferentes materias, podria ser interesante conocer a los mejores estudiantes, ya que vemos que el maximo en promedio general es 1.0, eso quiere decir que existen estudiantes con puntaje perfecto en todas las materias

In [25]:
#Mostrar los mejores 5 promedios generales
df.orderBy(col("general_average").desc()).show(5)

+-----+--------------+------------------+--------+-----------------------+---------------+------------------+------------------+---+------------------+
|index|race_ethnicity|parental_education|   lunch|test_preparation_course|math_percentage|reading_percentage|writing_percentage|sex|   general_average|
+-----+--------------+------------------+--------+-----------------------+---------------+------------------+------------------+---+------------------+
|  458|       group E| bachelor's degree|standard|                   none|            1.0|               1.0|               1.0|  F|               1.0|
|  916|       group E| bachelor's degree|standard|              completed|            1.0|               1.0|               1.0|  M|               1.0|
|  962|       group E|associate's degree|standard|                   none|            1.0|               1.0|               1.0|  F|               1.0|
|  114|       group E| bachelor's degree|standard|              completed|           0.9

De hecho existen 3 estudiantes con puntaje perfecto, que sucedera en el caso de los estudiantes con bajo puntaje?

In [26]:
#Mostrar los peores 5 promedios generales
df.orderBy(col("general_average").asc()).show(5)

+-----+--------------+------------------+------------+-----------------------+---------------+------------------+------------------+---+-------------------+
|index|race_ethnicity|parental_education|       lunch|test_preparation_course|math_percentage|reading_percentage|writing_percentage|sex|    general_average|
+-----+--------------+------------------+------------+-----------------------+---------------+------------------+------------------+---+-------------------+
|   59|       group C|  some high school|free/reduced|                   none|            0.0|              0.17|               0.1|  F|0.09000000000000001|
|  980|       group B|       high school|free/reduced|                   none|           0.08|              0.24|              0.23|  F|0.18333333333333335|
|  596|       group B|       high school|free/reduced|                   none|            0.3|              0.24|              0.15|  M|               0.23|
|  327|       group A|      some college|free/reduced|    

Podemos tomar algunos hallazgos de esto, no hay una tendencia clara con el sexo, pero si podemos ver alguna relacion con el tipo de comida y si completo el curso de preparacion para la prueba, tambien pareciera haber alguna relacion con el nivel de educacion de los padres, haremos los mas sencillos que serian los dos primeros (sera que el rector esta preocupado por la alimentacion de los estudiantes? o es mercenario y quiere vender mas cursos de preparacion?)

Utilizando la calculadora de muestras de es.surveymonkey.com, para una poblacion de 1000 se obtiene un resultado de 278 con nivel de confianza de 95% y margen de error 5%, tomaremos esa cantidad de alumnos de cada uno de los extremos para realizar los calculos. (Debe haber una metodologia mejor...)

In [27]:
#Primero obtengamos los tipos de desayuno disponibles, pudiera haber mas
df.select('lunch').distinct().show()

+------------+
|       lunch|
+------------+
|free/reduced|
|    standard|
+------------+



In [29]:
#Comprobamos que son dos, ahora obtengamos el porcentaje de desayunos standard para los 278 mas altos promedios
muestra_altoPromedio=df.orderBy(col("general_average").desc()).limit(278)
lunchStandard_altoPromedio= muestra_altoPromedio.filter(muestra_altoPromedio['lunch']=='standard').count()
pct_lunchStandard_altoPromedio= round((lunchStandard_altoPromedio/278)*100,2)

#Imprimir por pantalla el resultado
print(f'La cantidad de estudiantes con desayuno standard en los mejores 278 promedios es de {lunchStandard_altoPromedio}')
print(f'El porcentaje de estudiantes con desayuno standard en los 278 mejores promedios es de {pct_lunchStandard_altoPromedio}%')

La cantidad de estudiantes con desayuno standard en los mejores 278 promedios es de 216
El porcentaje de estudiantes con desayuno standard en los 278 mejores promedios es de 77.7%


In [30]:
#Obtenemos el porcentaje y cantidad de desayunos standard para los 278 promedios mas bajos
muestra_bajoPromedio= df.orderBy(col("general_average").asc()).limit(278)
lunchStandard_bajoPromedio= muestra_bajoPromedio.filter(muestra_bajoPromedio['lunch']=='standard').count()
pct_lunchStandard_bajoPromedio= round((lunchStandard_bajoPromedio/278)*100, 2)

#Imprimir por pantalla el resultado
print(f'La cantidad de estudiantes con desayuno standard en los peores 278 promedios es de {lunchStandard_bajoPromedio}')
print(f'El porcentaje de estudiantes con desayuno standard en los 278 mejores promedios es de {pct_lunchStandard_bajoPromedio}%')

La cantidad de estudiantes con desayuno standard en los peores 278 promedios es de 124
El porcentaje de estudiantes con desayuno standard en los 278 mejores promedios es de 44.6%


In [31]:
#Ahora revisaremos los resultados del curso de preparacion para el test
df.select('test_preparation_course').distinct().show()

+-----------------------+
|test_preparation_course|
+-----------------------+
|              completed|
|                   none|
+-----------------------+



In [32]:
#Solo existen dos resultados, repetimos la accion analizando los 278 mejores estudiantes y los 278 con el promedio mas bajo
preparationCourse_altoPromedio= muestra_altoPromedio.filter(muestra_altoPromedio['test_preparation_course']=='completed').count()
pct_preparationCourse_altoPromedio= round((preparationCourse_altoPromedio/278)*100, 2)

#Imprimir por pantalla el resultado
print(f'La cantidad de estudiantes que completaron el curso de preparacion en los mejores 278 promedios es de {preparationCourse_altoPromedio}')
print(f'El porcentaje de estudiantes que completaron el curso de preparacion en los mejores 278 promedios es de {pct_preparationCourse_altoPromedio}')

La cantidad de estudiantes que completaron el curso de preparacion en los mejores 278 promedios es de 137
El porcentaje de estudiantes que completaron el curso de preparacion en los mejores 278 promedios es de 49.28


In [33]:
#Repetimos la metodologia para los estudiantes con los promedios mas bajos
preparationCourse_bajoPromedio= muestra_bajoPromedio.filter(muestra_bajoPromedio['test_preparation_course']=='completed').count()
pct_preparationCourse_bajoPromedio= round((preparationCourse_bajoPromedio/278)*100, 2)

#Imprimir por pantalla el resultado
print(f'La cantidad de estudiantes que completaron el curso de preparacion en los peores 278 promedios es de {preparationCourse_bajoPromedio}')
print(f'El porcentaje de estudiantes que completaron el curso de preparacion en los peores 278 promedios es de {pct_preparationCourse_bajoPromedio}')

La cantidad de estudiantes que completaron el curso de preparacion en los peores 278 promedios es de 58
El porcentaje de estudiantes que completaron el curso de preparacion en los peores 278 promedios es de 20.86


### Sera la preocupacion del rector los desayunos? El curso de prepacion debe ser revisado? Lo descubriremos maniana en 'otro dia en la oficina llena de juntas'